In [ ]:
#installing needed libraries and models
%pip install insightface
%pip install onnxruntime-gpu
%pip install tqdm

In [27]:
# importing libraries
import os
import cv2
import numpy as np
from tqdm import tqdm
from insightface.app import FaceAnalysis

In [ ]:
# loading arcface
print("Loading ArcFace...")
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))
rec_model = app.models["recognition"]
print("ArcFace ready!")

In [ ]:
# extracting the embeddings using arcface
def save_embeddings_preserve_structure(image_root, embedding_root):
    total = 0

    for root, dirs, files in os.walk(image_root):
        rel_path = os.path.relpath(root, image_root)
        save_dir = os.path.join(embedding_root, rel_path)
        os.makedirs(save_dir, exist_ok=True)

        for file in files:

            if not file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue

            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)

            if img is None:
                print("Failed:", img_path)
                continue

            img = cv2.resize(img, (112, 112))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            embedding = rec_model.get_feat(img).flatten().astype(np.float32)
            save_path = os.path.join(save_dir, os.path.splitext(file)[0] + ".npy")
            np.save(save_path, embedding)
            total += 1

    print(f"Saved {total} embeddings")

In [ ]:
save_embeddings_preserve_structure(
    "dataset_split/train/gallery_cropped", "embeddings/train/gallery"
)

save_embeddings_preserve_structure(
    "dataset_split/test/gallery_cropped", "embeddings/test/gallery"
)

In [ ]:
save_embeddings_preserve_structure(
    "dataset_split/train/degraded_probes", "embeddings/train/degraded_probes"
)
save_embeddings_preserve_structure(
    "dataset_split/test/degraded_probes", "embeddings/test/degraded_probes"
)

In [ ]:
import numpy as np

sample = np.load(
    "/Users/admin/Desktop/reliable_rejection_under_degradation/second_half/embeddings/train/degraded_probes/Abdullah_Gul/Abdullah_Gul_0002_blur_high.npy"
)

print(sample.shape)
print(sample.dtype)

In [ ]:
# backup zip
import shutil

zip_path = shutil.make_archive("arcface_embeddings_backup", "zip", "embeddings")

print("Backup created:")
print(zip_path)